In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import holoviews as hv
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

sns.set_theme()
sns.despine()

tqdm.pandas()
hv.extension('bokeh')

%matplotlib inline

In [ ]:
INTER_DATA_PATH = "../../intermediary_data/"
filename_csv = "lab.csv"
filename_parq = "lab.parquet"
file_path_csv = Path(INTER_DATA_PATH) / Path(filename_csv)
file_path_parq = Path(INTER_DATA_PATH) / Path(filename_parq)

In [ ]:
mini_sample_size = True

if mini_sample_size:
    df = pd.read_parquet(path=file_path_parq, engine="pyarrow").head(n=1000)
else:
    df = pd.read_parquet(path=file_path_parq, engine="pyarrow")

In [ ]:
df.info()

In [ ]:
def convert_to_timedelta(stay):
    stay["time_delta"] = pd.to_timedelta(stay["labresultoffset"] - stay["labresultoffset"].iloc[0], unit="min")
    stay["time_delta_revised"] = pd.to_timedelta(
        stay["labresultrevisedoffset"] - stay["labresultrevisedoffset"].iloc[0], unit="min")

    return stay

In [ ]:
df.head()

fig_hist, ax = plt.subplots(figsize=(35, 15))

sns.countplot(data=df, x="labname", hue="labname", ax=ax, stat="count", orient="y", palette="plasma", legend=False)

plt.xticks(rotation=90)

Bedside glucose measurements dominate the occurances due to fact that they are measured at least once a day for diabetic patients.

The parameters belonging to the group of blond panel markers (CBC), e.g. BUN, HGB, RBC, WBC, bicarbonat, calcium, creatinin, platelets, sodium, etc. are the second most common ones - since they are the "mainstay" of in-vitro diagnostics and every patient will get blood tests.
Roughly half of the occurrences are in turn blood gas analysis parameters and more detailed immunophenotyping parameters like the cell counts of individual adaptive IS cell families.

The "rare" lab test are most probably test that are conducted based on the condition of the patient and the consideration of the clinicians. E.g. troponin when trying to detect damaged heart tissue.

Note that the term "rare" is kinda relative here, since the y-axis scale magnitude is 1e6.

The targeted test might contain additional information or knowledge with regard to the state of the patient, since clinicans usually don't issue those tests for no reason.
While the routine lab test might be more suitable for anomaly detection.


In [ ]:
def min_to_hours(x, pos):
    total_seconds = 60 * int(x)
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60

    return f"{hours}:{minutes:02d}"


min_hour_formatter = ticker.FuncFormatter(func=min_to_hours)

df["patientunitstayid"] = df["patientunitstayid"].astype(int)





fig_time_scatter, ax = plt.subplots(figsize=(20, 30)) 

ax.xaxis.set_major_formatter(min_hour_formatter)
sns.scatterplot(data=df, x="labresultoffset", y="labname", hue="patientunitstayid", #palette="inferno", ax=ax)



There are plenty of weird lab measurement results which reach extremely long back, those might be faulty data points.
It might be beneficial to exclude lab results that are prior to the ICU admission.

In [ ]:
#fig_hist.savefig("lab_plot_outputs/lab_hist.png", dpi=300)
#fig_time_scatter.savefig("lab_plot_outputs/lab_ts_scatter.png", dpi=300)

fig_time_scatter_lim, ax = plt.subplots(figsize=(20, 30))

min_hour_formatter = ticker.FuncFormatter(func=min_to_hours)
ax.xaxis.set_major_formatter(min_hour_formatter)

sns.scatterplot(data=df[df["labresultoffset"] > 0], x="labresultoffset", y="labname", hue="patientunitstayid",
                palette="inferno", ax=ax, legend="auto")


In [ ]:
#fig_time_scatter_lim.savefig("lab_plot_outputs/lab_ts_scatter_from_start.png", dpi=300)

In [ ]:
from matplotlib.ticker import MultipleLocator

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})


start_time = 0

days = 7
hours = days * 24
minutes = hours * 60

#num_columns = len(df.columns)
num_columns = 160
#pal = sns.cubehelix_palette(num_columns, rot=-.25, light=.7)
pal = sns.color_palette("plasma")

In [ ]:
grid = sns.FacetGrid(df[(df["labresultoffset"] <= minutes) & (df["labresultoffset"] > start_time)], row="labname",
                     #grid = sns.FacetGrid(small_df[(small_df["labresultoffset"] <= minutes) & (small_df["labresultoffset"] > start_time)], row="labname",
                     hue="labname", aspect=15, height=1, palette=pal)

grid.map_dataframe(sns.kdeplot, "labresultoffset",
                   #bw_adjust=.5,
                   #bw_adjust=.1,
                   bw_adjust=.05,
                   clip_on=False,
                   fill=True,
                   alpha=1,
                   linewidth=1.5
                   )

# white contour around kde
grid.map(sns.kdeplot, "labresultoffset", clip_on=False, color="w", lw=1,
         #bw_adjust=.5
         #bw_adjust=.1
         bw_adjust=.05
         )

grid.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)


def label(x, color, label):
    ax = plt.gca()
    #ax.text(0, .2, label, fontweight="bold", color=color,
    ax.text(
        -0.1,
        0.2,
        label, fontweight="bold", color=color,
        ha="left", va="center", transform=ax.transAxes)


grid.map(label, "labname")
grid.figure.subplots_adjust(hspace=-.5)

# removing all the clutter, label are already dealt with by the label map
grid.set_titles("")
grid.set(yticks=[], ylabel="")
grid.set(xlabel="Time [hours:minutes]")
grid.despine(bottom=True, left=True)

for ax in grid.axes.flat:
    minutes_per_day = 24 * 60
    ax.xaxis.set_major_locator(MultipleLocator(minutes_per_day))
    ax.xaxis.set_major_formatter(min_hour_formatter)
    ax.grid(which="major", axis="x", linestyle="-", color="grey")
    ax.set_axisbelow(True)
    ax.grid(zorder=0, alpha=0.3)
    ax.set_zorder(3)